In [1]:
import sys; sys.path.insert(0, "../src")
from ds_prep.stats_utils import two_prop_ztest, welch_ttest, bootstrap_ci

from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

from scipy import stats

import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("../data/experiment.csv")
df.head()

,user_id,variant,exposure_date,pre_gmv_28d,orders_14d,gmv_14d
0,14357,treatment,2026-09-04,1534.52,4,1441.69
1,3440,control,2026-09-04,1817.86,1,368.36
2,12154,control,2026-09-01,0.00,1,421.78
3,15030,treatment,2026-09-06,0.00,1,258.44
4,18550,treatment,2026-09-01,518.17,1,166.94


In [3]:
df["is_converted_user"] = df["orders_14d"] > 0

### Two-proportions z-test

In [4]:
x1 = len(df[(df["variant"] == "treatment") & (df["is_converted_user"])])
n1 = len(df[df["variant"] == "treatment"])
x2 = len(df[(df["variant"] == "control") & (df["is_converted_user"])])
n2 = len(df[df["variant"] == "control"])

In [5]:
z_act, p_value_act = proportions_ztest(count=[x1,x2], nobs=[n1, n2])
ci_low_act, ci_high_act = confint_proportions_2indep(
    x1, n1, x2, n2,
    method = "wald",
    compare = "diff"
)

pred_z_test_result = two_prop_ztest(x1, n1, x2, n2)
z_pred = pred_z_test_result["z"]
p_value_pred = pred_z_test_result["p_value"]
ci_low_pred = pred_z_test_result["ci_low"]
ci_high_pred = pred_z_test_result["ci_high"]

if round(z_act, 4) == round(z_pred, 4):
    z_match = "✅"
else:
    z_match = "❌"

if round(p_value_act, 4) == round(p_value_pred, 4):
    p_value_match = "✅"
else:
    p_value_match = "❌"

if round(ci_low_act, 4) == round(ci_low_pred, 4):
    ci_low_match = "✅"
else:
    ci_low_match = "❌"

if round(ci_high_act, 4) == round(ci_high_pred, 4):
    ci_high_match = "✅"
else:
    ci_high_match = "❌"

print(f"z{z_match}, actual: {z_act:.4f}, predicted: {z_pred:.4f}")
print(f"p_value{p_value_match}, actual: {p_value_act:.4f}, predicted: {p_value_pred:.4f}")
print(f"ci_low{ci_low_match}, actual: {ci_low_act:.4f}, predicted: {ci_low_pred:.4f}")
print(f"ci_high{ci_high_match}, actual: {ci_high_act:.4f}, predicted: {ci_high_pred:.4f}")

z✅, actual: 4.1852, predicted: 4.1852
p_value✅, actual: 0.0000, predicted: 0.0000
ci_low✅, actual: 0.0198, predicted: 0.0198
ci_high✅, actual: 0.0546, predicted: 0.0546


### Welch's two-sample t-test

In [6]:
a = list(df[df["variant"] == "treatment"]["gmv_14d"])
b = list(df[df["variant"] == "control"]["gmv_14d"])

res = stats.ttest_ind(a, b, equal_var=False)

t_act = res.statistic
df_act = res.df
p_value_act = res.pvalue
ci = res.confidence_interval(confidence_level=0.95)
ci_low_act = ci.low
ci_high_act = ci.high

pred_welch_test_result = welch_ttest(a, b)
t_pred = pred_welch_test_result["t"]
df_pred = pred_welch_test_result["df"]
p_value_pred = pred_welch_test_result["p_value"]
ci_low_pred = pred_welch_test_result["ci_low"]
ci_high_pred = pred_welch_test_result["ci_high"]

if round(t_act, 4) == round(t_pred, 4):
    t_match = "✅"
else:
    t_match = "❌"

if round(df_act, 4) == round(df_pred, 4):
    df_match = "✅"
else:
    df_match = "❌"

if round(p_value_act, 4) == round(p_value_pred, 4):
    p_value_match = "✅"
else:
    p_value_match = "❌"

if round(ci_low_act, 4) == round(ci_low_pred, 4):
    ci_low_match = "✅"
else:
    ci_low_match = "❌"

if round(ci_high_act, 4) == round(ci_high_pred, 4):
    ci_high_match = "✅"
else:
    ci_high_match = "❌"

print(f"t{t_match}, actual: {t_act:.4f}, predicted: {t_pred:.4f}")
print(f"t{df_match}, actual: {df_act:.4f}, predicted: {df_pred:.4f}")
print(f"p_value{p_value_match}, actual: {p_value_act:.4f}, predicted: {p_value_pred:.4f}")
print(f"ci_low{ci_low_match}, actual: {ci_low_act:.4f}, predicted: {ci_low_pred:.4f}")
print(f"ci_high{ci_high_match}, actual: {ci_high_act:.4f}, predicted: {ci_high_pred:.4f}")

t✅, actual: 5.0730, predicted: 5.0730
t✅, actual: 11995.4637, predicted: 11995.4637
p_value✅, actual: 0.0000, predicted: 0.0000
ci_low✅, actual: 33.6589, predicted: 33.6589
ci_high✅, actual: 76.0496, predicted: 76.0496


### Bootstrap

In [7]:
def mean_diff(x, y, axis):
    return np.mean(x, axis=axis) - np.mean(y, axis=axis)

res = stats.bootstrap(
    (a, b),                       # data: a tuple, one entry per group
    mean_diff,                    # your statistic
    n_resamples=10_000,
    confidence_level=0.95,
    method="percentile",          # match YOUR method; the default is "BCa"
    rng=np.random.default_rng(0), # seed
)

ci_low_act = res.confidence_interval.low
ci_high_act = res.confidence_interval.high

pred_boostrap_ci_result = bootstrap_ci(a, b)
ci_low_pred = pred_boostrap_ci_result["ci_low"]
ci_high_pred = pred_boostrap_ci_result["ci_high"]

if round(ci_low_act, 4) == round(ci_low_pred, 4):
    ci_low_match = "✅"
else:
    ci_low_match = "❌"

if round(ci_high_act, 4) == round(ci_high_pred, 4):
    ci_high_match = "✅"
else:
    ci_high_match = "❌"

print(f"ci_low{ci_low_match}, actual: {ci_low_act:.4f}, predicted: {ci_low_pred:.4f}")
print(f"ci_high{ci_high_match}, actual: {ci_high_act:.4f}, predicted: {ci_high_pred:.4f}")

ci_low✅, actual: 33.8107, predicted: 33.8107
ci_high✅, actual: 76.0052, predicted: 76.0052
